# SpatialFlow backend — Kaggle launcher

Notebook này khởi tạo backend cho pipeline:

**Gemini → Scene Graph → Layout → SD3.5 → GroundingDINO + SAM2 → TRELLIS → GLB scene**

## Cách chạy

Chạy tuần tự từ trên xuống. Không chạy các cell cài đặt cũ từ notebook khác vào cùng session.

- Python 3.10 được đặt tại `/opt/venv310`.
- Source backend được đồng bộ vào `/kaggle/working`.
- Secrets được lấy từ Kaggle Secrets hoặc nhập ẩn bằng `getpass`.
- Cell preflight phải báo `BACKEND PREFLIGHT PASSED` trước khi mở server.
- Cell khởi động tự dừng server cũ nên có thể chạy lại an toàn.


## 1. Cấu hình và secrets

Trong Kaggle, tạo bốn Secrets: `HF_TOKEN`, `GEMINI_API_KEY`, `NGROK_TOKEN`, `APP_API_KEY`.
Notebook không ghi giá trị token vào output.


In [ ]:
import os
from getpass import getpass


def read_secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value

    try:
        from kaggle_secrets import UserSecretsClient
        value = (UserSecretsClient().get_secret(name) or "").strip()
        if value:
            return value
    except Exception:
        pass

    return getpass(f"Nhap {name}: ").strip()


HF_TOKEN = read_secret("HF_TOKEN")
GEMINI_API_KEY = read_secret("GEMINI_API_KEY")
NGROK_TOKEN = read_secret("NGROK_TOKEN")
APP_API_KEY = read_secret("APP_API_KEY")
GEMINI_MODEL = "gemini-3.1-flash-lite"
SD35_LORA_PATH = os.environ.get("SD35_LORA_PATH", "/kaggle/working/lora_sd35_fast_safe/best").strip()
SD35_LORA_SCALE = "0.2"

required_secrets = {
    "HF_TOKEN": HF_TOKEN,
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "NGROK_TOKEN": NGROK_TOKEN,
    "APP_API_KEY": APP_API_KEY,
}
missing = [name for name, value in required_secrets.items() if not value]
if missing:
    raise RuntimeError("Thieu secret: " + ", ".join(missing))

for name, value in required_secrets.items():
    os.environ[name] = value
os.environ["GEMINI_MODEL"] = GEMINI_MODEL
os.environ["SD35_LORA_PATH"] = SD35_LORA_PATH
os.environ["SD35_LORA_SCALE"] = SD35_LORA_SCALE
os.environ.setdefault("HF_HOME", "/kaggle/working/hf_cache")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("Secrets loaded:", {name: bool(value) for name, value in required_secrets.items()})
print("Gemini model:", GEMINI_MODEL)


## 2. Tạo Python 3.10 virtual environment

Cell này chỉ cài Python hệ thống khi cần. Đặt `REBUILD_VENV=True` nếu muốn dựng lại môi trường từ đầu.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

VENV = Path("/opt/venv310")
PYTHON = VENV / "bin/python"
PIP = VENV / "bin/pip"
REBUILD_VENV = False


def run(command, *, env=None, timeout=1800, check=True):
    print("+", " ".join(map(str, command)))
    result = subprocess.run(
        list(map(str, command)),
        env=env,
        timeout=timeout,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0 and result.stderr:
        print(result.stderr[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(map(str, command))}"
        )
    return result


if REBUILD_VENV and VENV.exists():
    shutil.rmtree(VENV)

# Kaggle may expose python3.10 but omit ensurepip/python3.10-venv.
# Install the system package unconditionally before creating the venv.
run(["apt-get", "update", "-qq"])
run([
    "apt-get", "install", "-y", "-qq",
    "software-properties-common",
])
run(["add-apt-repository", "ppa:deadsnakes/ppa", "-y"])
run(["apt-get", "update", "-qq"])
run([
    "apt-get", "install", "-y", "-qq",
    "python3.10", "python3.10-dev", "python3.10-venv",
    "build-essential", "git", "git-lfs", "wget",
])

python310 = shutil.which("python3.10")
if not python310:
    raise RuntimeError("Khong tim thay Python 3.10 sau khi cai dat")

if not PYTHON.is_file():
    run([python310, "-m", "venv", str(VENV)])

run([str(PYTHON), "--version"])
run([
    str(PYTHON), "-m", "pip", "install", "-q", "--upgrade",
    "pip==24.0", "setuptools==69.5.1", "wheel", "ninja",
])
print("Virtual environment ready:", VENV)


## 3. Đồng bộ source backend

Chỉ các file ứng dụng cần thiết được chép vào `/kaggle/working`, tránh để file cũ còn sót lại.


In [ ]:
import shutil
from pathlib import Path

REPOSITORY_URL = "https://github.com/Tiens0710/DATN-3d.git"
REPOSITORY_REF = "main"
CLONE_ROOT = Path("/tmp/spatialflow_backend")
WORKING_ROOT = Path("/kaggle/working")

if CLONE_ROOT.exists():
    shutil.rmtree(CLONE_ROOT)

run([
    "git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
    REPOSITORY_URL, str(CLONE_ROOT),
])

required_source = [
    CLONE_ROOT / "server.py",
    CLONE_ROOT / "worker_sd35.py",
    CLONE_ROOT / "worker_sam2_dino.py",
    CLONE_ROOT / "worker_trellis.py",
    CLONE_ROOT / "requirements.txt",
    CLONE_ROOT / "src",
    CLONE_ROOT / "src" / "parser.py",
]
missing_source = [str(path) for path in required_source if not path.exists()]
if missing_source:
    raise FileNotFoundError("Source repository thieu: " + ", ".join(missing_source))

for name in (
    "server.py", "worker_sd35.py", "worker_sam2_dino.py",
    "worker_trellis.py", "requirements.txt",
):
    shutil.copy2(CLONE_ROOT / name, WORKING_ROOT / name)

target_src = WORKING_ROOT / "src"
if target_src.exists():
    shutil.rmtree(target_src)
shutil.copytree(CLONE_ROOT / "src", target_src)

print("Backend source synchronized:", WORKING_ROOT)


## 4. Cài một bộ dependency duy nhất

Không chạy thêm các cell đổi phiên bản Diffusers/Transformers/PEFT sau cell này.


In [ ]:
TORCH_INDEX = "https://download.pytorch.org/whl/cu121"

run([
    str(PYTHON), "-m", "pip", "install", "-q", "--no-cache-dir",
    "torch==2.1.0", "torchvision==0.16.0", "xformers==0.0.22.post7",
    "--index-url", TORCH_INDEX,
])

packages = [
    "numpy==1.26.4",
    "diffusers==0.32.2",
    "transformers==4.46.3",
    "peft==0.10.0",
    "accelerate==0.34.2",
    "huggingface_hub==0.34.6",
    "safetensors>=0.4.5",
    "fastapi",
    "pydantic",
    "python-multipart",
    "uvicorn",
    "pyngrok",
    "requests",
    "spacy>=3.5.0",
    "hydra-core>=1.3.2",
    "iopath>=0.1.10",
    "timm==0.9.16",
    "supervision",
    "pycocotools",
    "addict",
    "yapf",
    "trimesh==4.5.3",
    "plyfile==0.9",
    "pymeshfix",
    "pyvista==0.44.2",
    "igraph==0.11.8",
    "xatlas==0.0.9",
    "scipy==1.14.1",
    "pillow==10.4.0",
    "imageio==2.36.1",
    "imageio-ffmpeg==0.5.1",
    "opencv-python-headless==4.10.0.84",
    "tqdm==4.67.1",
    "easydict==1.13",
    "rembg[cpu]==2.0.60",
    "spconv-cu121==2.3.8",
    "setuptools==69.5.1",
    "wheel",
    "ninja",
]
run([
    str(PYTHON), "-m", "pip", "install", "-q", "--no-cache-dir",
    *packages,
])

run([
    str(PYTHON), "-m", "pip", "install", "-q", "--no-cache-dir",
    "--no-build-isolation",
    "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8",
])

run([str(PYTHON), "-m", "spacy", "download", "en_core_web_sm"])
print("Pinned Python dependencies installed.")


## 5. GroundingDINO, SAM2 và checkpoints


In [ ]:
import os
import shutil
from pathlib import Path
from urllib.request import urlretrieve

VENDOR_ROOT = Path("/kaggle/working/vendor")
VENDOR_ROOT.mkdir(parents=True, exist_ok=True)


def fresh_clone(url, destination):
    destination = Path(destination)
    if destination.exists():
        shutil.rmtree(destination)
    run(["git", "clone", "--depth", "1", url, str(destination)])
    return destination


grounding_root = fresh_clone(
    "https://github.com/IDEA-Research/GroundingDINO.git",
    VENDOR_ROOT / "GroundingDINO",
)
sam2_root = fresh_clone(
    "https://github.com/facebookresearch/sam2.git",
    VENDOR_ROOT / "sam2",
)

build_env = os.environ.copy()
build_env["CUDA_HOME"] = "/usr/local/cuda"
build_env["PATH"] = "/usr/local/cuda/bin:" + build_env.get("PATH", "")

run([
    str(PYTHON), "-m", "pip", "install", "-q",
    "--no-deps", "--no-build-isolation", "-e", str(grounding_root),
], env=build_env)
run([
    str(PYTHON), "-m", "pip", "install", "-q",
    "--no-deps", "--no-build-isolation", "-e", str(sam2_root),
], env=build_env)

grounding_ckpt = Path("/kaggle/working/groundingdino_ckpt")
sam2_ckpt = Path("/kaggle/working/sam2_ckpt")
grounding_ckpt.mkdir(parents=True, exist_ok=True)
sam2_ckpt.mkdir(parents=True, exist_ok=True)

downloads = {
    grounding_ckpt / "groundingdino_swint_ogc.pth":
        "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth",
    sam2_ckpt / "sam2_hiera_small.pt":
        "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt",
}
for destination, url in downloads.items():
    if not destination.is_file() or destination.stat().st_size < 1_000_000:
        print("Downloading:", destination.name)
        urlretrieve(url, destination)
    if destination.stat().st_size < 1_000_000:
        raise RuntimeError(f"Checkpoint khong hop le: {destination}")

shutil.copy2(
    grounding_root / "groundingdino/config/GroundingDINO_SwinT_OGC.py",
    grounding_ckpt / "GroundingDINO_SwinT_OGC.py",
)
print("GroundingDINO and SAM2 ready.")


## 6. TRELLIS và CUDA extensions

Các extension được cài đúng một lần tại đây, không cài lại trong request API.


In [ ]:
import os
import shutil
from pathlib import Path

TRELLIS_ROOT = Path("/kaggle/working/TRELLIS")
if TRELLIS_ROOT.exists():
    shutil.rmtree(TRELLIS_ROOT)

clone_env = os.environ.copy()
clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
run([
    "git", "clone",
    "https://huggingface.co/spaces/trellis-community/TRELLIS",
    str(TRELLIS_ROOT),
], env=clone_env)

build_env = os.environ.copy()
build_env["CUDA_HOME"] = "/usr/local/cuda"
build_env["PATH"] = "/usr/local/cuda/bin:" + build_env.get("PATH", "")
build_env["TORCH_CUDA_ARCH_LIST"] = "7.5"
build_env["MAX_JOBS"] = "2"

run([
    str(PYTHON), "-m", "pip", "install", "-q",
    "--no-deps", "--no-cache-dir", "--no-build-isolation",
    "git+https://github.com/NVlabs/nvdiffrast.git",
], env=build_env, timeout=1200)

mip_root = Path("/tmp/mip-splatting")
if mip_root.exists():
    shutil.rmtree(mip_root)
run([
    "git", "clone", "--recursive", "--depth", "1",
    "https://github.com/autonomousvision/mip-splatting.git",
    str(mip_root),
])
run([
    str(PYTHON), "-m", "pip", "install", "-q",
    "--no-deps", "--no-cache-dir", "--no-build-isolation",
    str(mip_root / "submodules/diff-gaussian-rasterization"),
], env=build_env, timeout=1200)

print("TRELLIS and CUDA extensions installed.")


## 7. Preflight bắt buộc

Cell này kiểm tra phiên bản, CUDA, token, checkpoint, source và import thực tế. Không mở server nếu cell này lỗi.


In [ ]:
import json
import os
import subprocess
from pathlib import Path

required_files = [
    Path("/kaggle/working/server.py"),
    Path("/kaggle/working/worker_sd35.py"),
    Path("/kaggle/working/worker_sam2_dino.py"),
    Path("/kaggle/working/worker_trellis.py"),
    Path("/kaggle/working/src/generator_2d.py"),
    Path("/kaggle/working/src/generator_3d.py"),
    Path("/kaggle/working/groundingdino_ckpt/groundingdino_swint_ogc.pth"),
    Path("/kaggle/working/groundingdino_ckpt/GroundingDINO_SwinT_OGC.py"),
    Path("/kaggle/working/sam2_ckpt/sam2_hiera_small.pt"),
    Path("/kaggle/working/TRELLIS/trellis"),
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Preflight thieu file:\n" + "\n".join(missing_files))

preflight_script = r"""
import json
import os
import sys

sys.path.insert(0, "/kaggle/working")
sys.path.insert(0, "/kaggle/working/TRELLIS")
sys.modules["triton"] = None

import torch
import torchvision
import numpy
import diffusers
import transformers
import peft
import accelerate
import xformers
import spconv
import nvdiffrast.torch
import utils3d
import groundingdino
from pycocotools import mask as pycocotools_mask
import multipart
import sam2
import trimesh
from diff_gaussian_rasterization import _C
from trellis.pipelines import TrellisImageTo3DPipeline
import server
from src.parser import parse_scene_graph

parser_probe = parse_scene_graph("one wooden dining table and one wooden dining chair beside the table")
parser_labels = [node["label"] for node in parser_probe.get("nodes", [])]
if parser_labels != ["table", "chair"]:
    raise RuntimeError(f"Parser probe failed: expected ['table', 'chair'], got {parser_labels}")
print("Parser probe passed: one table + one chair")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available")
if not os.environ.get("HF_TOKEN", "").strip():
    raise RuntimeError("HF_TOKEN is missing")

versions = {
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": numpy.__version__,
    "diffusers": diffusers.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "accelerate": accelerate.__version__,
    "xformers": xformers.__version__,
    "cuda": torch.cuda.get_device_name(0),
}
print(json.dumps(versions, indent=2))
"""

preflight_env = os.environ.copy()
preflight_env["HF_TOKEN"] = HF_TOKEN
preflight_env["GEMINI_API_KEY"] = GEMINI_API_KEY
preflight_env["GEMINI_MODEL"] = GEMINI_MODEL
preflight_env["SPCONV_ALGO"] = "native"
preflight_env["ATTN_BACKEND"] = "xformers"
preflight_env["SPARSE_ATTN"] = "xformers"
preflight_env["MPLBACKEND"] = "agg"

result = run(
    [str(PYTHON), "-c", preflight_script],
    env=preflight_env,
    timeout=300,
    check=False,
)
if result.returncode != 0:
    raise RuntimeError("BACKEND PREFLIGHT FAILED")

print("BACKEND PREFLIGHT PASSED")


## 8. Dừng tiến trình cũ và khởi động FastAPI


In [ ]:
import json
import os
import subprocess
import time
from pathlib import Path

import requests

WORKERS = {
    "sd35": {
        "script": Path("/kaggle/working/worker_sd35.py"),
        "url": "http://127.0.0.1:8001",
        "log": Path("/kaggle/working/sd35_worker.log"),
        "timeout": 900,
    },
    "sam2_dino": {
        "script": Path("/kaggle/working/worker_sam2_dino.py"),
        "url": "http://127.0.0.1:8003",
        "log": Path("/kaggle/working/sam2_dino_worker.log"),
        "timeout": 300,
    },
    "trellis": {
        "script": Path("/kaggle/working/worker_trellis.py"),
        "url": "http://127.0.0.1:8002",
        "log": Path("/kaggle/working/trellis_worker.log"),
        "timeout": 900,
    },
}
SERVER = Path("/kaggle/working/server.py")
LOG_PATH = Path("/kaggle/working/server.log")

def stop_process(process_name, process_attr):
    try:
        process = globals().get(process_attr)
        if process is not None and process.poll() is None:
            process.terminate()
            process.wait(timeout=10)
    except Exception:
        pass
    subprocess.run(["pkill", "-f", process_name], check=False)

for worker_name, worker_config in WORKERS.items():
    stop_process(
        str(worker_config["script"]),
        f"{worker_name}_worker_proc",
    )
stop_process("/kaggle/working/server.py", "server_proc")
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
time.sleep(2)

worker_env = os.environ.copy()
worker_env["PYTHONUNBUFFERED"] = "1"
worker_env["HF_TOKEN"] = HF_TOKEN
worker_env["SD35_LORA_PATH"] = SD35_LORA_PATH
worker_env["SD35_LORA_SCALE"] = SD35_LORA_SCALE
worker_env["SPCONV_ALGO"] = "native"
worker_env["ATTN_BACKEND"] = "xformers"
worker_env["SPARSE_ATTN"] = "xformers"
worker_env["MPLBACKEND"] = "agg"
worker_env["ENABLE_MATTING"] = "1"
worker_env["MATTING_MODEL"] = "isnet-general-use"
worker_processes = {}
try:
    for worker_name, worker_config in WORKERS.items():
        worker_log = open(
            worker_config["log"], "w", buffering=1, encoding="utf-8"
        )
        worker_port = worker_config["url"].rsplit(":", 1)[-1]
        worker_proc = subprocess.Popen(
            [
                str(PYTHON), "-u", str(worker_config["script"]),
                "--port", worker_port,
            ],
            cwd="/kaggle/working",
            stdout=worker_log,
            stderr=subprocess.STDOUT,
            env=worker_env,
        )
        globals()[f"{worker_name}_worker_proc"] = worker_proc
        globals()[f"{worker_name}_worker_log"] = worker_log
        worker_processes[worker_name] = worker_proc

        worker_health = None
        for second in range(worker_config["timeout"]):
            if worker_proc.poll() is not None:
                worker_log.flush()
                raise RuntimeError(
                    f"{worker_name} worker da dung:\n"
                    + worker_config["log"].read_text(
                        encoding="utf-8", errors="replace"
                    )[-8000:]
                )
            try:
                response = requests.get(
                    worker_config["url"] + "/health", timeout=2
                )
                if response.ok:
                    current_health = response.json()
                    if current_health.get("ready"):
                        worker_health = current_health
                        break
                    if current_health.get("error"):
                        raise RuntimeError(current_health["error"])
            except requests.RequestException:
                pass
            if (second + 1) % 10 == 0:
                print(
                    f"Dang cho {worker_name} worker... "
                    f"{second + 1}/{worker_config['timeout']}s"
                )
            time.sleep(1)

        if worker_health is None:
            raise RuntimeError(
                f"{worker_name} worker khong san sang:\n"
                + worker_config["log"].read_text(
                    encoding="utf-8", errors="replace"
                )[-8000:]
            )
        print(json.dumps(worker_health, ensure_ascii=False, indent=2))
        print(f"{worker_name.upper()} WORKER READY")
        if worker_name == "sd35":
            offload_response = requests.post(
                worker_config["url"] + "/offload", timeout=300
            )
            offload_response.raise_for_status()
            offload_status = offload_response.json()
            if offload_status.get("device") != "cpu":
                raise RuntimeError(
                    "SD3.5 worker did not release GPU memory: "
                    + json.dumps(offload_status, ensure_ascii=False)
                )
            print("SD3.5 OFFLOADED TO CPU - VRAM RELEASED")
except Exception:
    for started_name, started_proc in worker_processes.items():
        if started_proc.poll() is None:
            started_proc.terminate()
    raise

subprocess.run(["nvidia-smi"], check=False)

server_env = worker_env.copy()
server_env["GEMINI_API_KEY"] = GEMINI_API_KEY
server_env["GEMINI_MODEL"] = GEMINI_MODEL
server_env["APP_API_KEY"] = APP_API_KEY
server_env["SD35_WORKER_URL"] = WORKERS["sd35"]["url"]
server_env["SAM2_DINO_WORKER_URL"] = WORKERS["sam2_dino"]["url"]
server_env["TRELLIS_WORKER_URL"] = WORKERS["trellis"]["url"]
server_log = open(LOG_PATH, "w", buffering=1, encoding="utf-8")
server_proc = subprocess.Popen(
    [str(PYTHON), "-u", str(SERVER)],
    cwd="/kaggle/working",
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=server_env,
)

health = None
for second in range(120):
    if server_proc.poll() is not None:
        server_log.flush()
        raise RuntimeError(
            "Server da dung:\n"
            + LOG_PATH.read_text(encoding="utf-8", errors="replace")[-6000:]
        )
    try:
        response = requests.get("http://127.0.0.1:8000/api/health", timeout=3)
        if response.ok:
            health = response.json()
            break
    except requests.RequestException:
        pass
    if (second + 1) % 10 == 0:
        print(f"Dang cho FastAPI... {second + 1}/120")
    time.sleep(1)

if health is None:
    raise RuntimeError(
        "FastAPI khong san sang:\n"
        + LOG_PATH.read_text(encoding="utf-8", errors="replace")[-6000:]
    )

print(json.dumps(health, ensure_ascii=False, indent=2))
if health.get("ready") is False:
    raise RuntimeError("FastAPI chay nhung backend readiness=false")
print("FASTAPI READY")


## 9. Smoke test endpoint nhẹ

Không sinh ảnh hoặc GLB; chỉ kiểm tra prompt optimizer, parser và layout.


In [ ]:
import requests

BASE_URL = "http://127.0.0.1:8000"

worker_urls = {
    "sd35": "http://127.0.0.1:8001",
    "trellis": "http://127.0.0.1:8002",
    "sam2_dino": "http://127.0.0.1:8003",
}
for worker_name, worker_url in worker_urls.items():
    worker_response = requests.get(worker_url + "/health", timeout=5)
    worker_response.raise_for_status()
    worker_health = worker_response.json()
    if not worker_health.get("ready"):
        raise RuntimeError(
            f"{worker_name} worker is not ready: {worker_health}"
        )
    print(worker_name, "worker health OK")

tests = [
    ("health", "GET", "/api/health", None),
    (
        "optimize_prompt",
        "POST",
        "/api/optimize_prompt",
        {"prompt": "mot cai ghe go"},
    ),
    (
        "parse_scene_graph",
        "POST",
        "/api/parse_scene_graph",
        {"text": "one wooden table beside one wooden chair"},
    ),
]

responses = {}
for name, method, endpoint, payload in tests:
    response = requests.request(
        method,
        BASE_URL + endpoint,
        json=payload,
        timeout=60,
    )
    response.raise_for_status()
    responses[name] = response.json()
    print(name, "OK")

scene_graph = responses["parse_scene_graph"]
layout_response = requests.post(
    BASE_URL + "/api/generate_layout",
    json=scene_graph,
    timeout=30,
)
layout_response.raise_for_status()
print("generate_layout OK")
print("BACKEND SMOKE TEST PASSED")


## 10. Mở Ngrok

Sao chép giá trị `API URL` vào ô **Ngrok server URL** trên giao diện SpatialFlow.


In [ ]:
import time
from pathlib import Path

import requests
from pyngrok import ngrok

LOCAL_API_URL = "http://127.0.0.1:8000"
SERVER_LOG = Path("/kaggle/working/server.log")
local_health = None

# Khong mo Ngrok khi FastAPI chua song; neu khong, Ngrok se tra 502.
for second in range(120):
    try:
        local_response = requests.get(
            LOCAL_API_URL + "/api/health",
            timeout=3,
        )
        if local_response.ok:
            local_health = local_response.json()
            break
    except requests.RequestException:
        pass

    if (second + 1) % 10 == 0:
        print(f"Dang cho FastAPI truoc khi mo Ngrok... {second + 1}/120s")
    time.sleep(1)

if local_health is None:
    log_tail = (
        SERVER_LOG.read_text(encoding="utf-8", errors="replace")[-8000:]
        if SERVER_LOG.exists()
        else "Khong tim thay server.log"
    )
    raise RuntimeError("FastAPI chua san sang:\n" + log_tail)

if local_health.get("ready") is False:
    raise RuntimeError("FastAPI readiness=false:\n" + str(local_health))

try:
    ngrok.kill()
except Exception:
    pass
time.sleep(1)

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(addr="127.0.0.1:8000", proto="http")
PUBLIC_API_URL = tunnel.public_url.rstrip("/")

response = requests.get(
    PUBLIC_API_URL + "/api/health",
    headers={"ngrok-skip-browser-warning": "69420"},
    timeout=30,
)
response.raise_for_status()
public_health = response.json()
if public_health.get("ready") is False:
    raise RuntimeError("Public API readiness=false:\n" + str(public_health))

print("API URL:", PUBLIC_API_URL)
print("Health ready:", public_health.get("ready", "not-reported"))


## 11. Theo dõi log backend và GPU

Cell dưới hiển thị trạng thái process, VRAM và phần cuối log của FastAPI/ba worker. Đặt `FOLLOW_LIVE = True` để tự làm mới mỗi 2 giây; dừng bằng nút **Stop** của Kaggle hoặc `Ctrl+C`. Khi TRELLIS lỗi, xem mục `trellis` để lấy traceback đầy đủ.


In [ ]:
import subprocess
import requests
import time
from datetime import datetime
from pathlib import Path

from IPython.display import clear_output

LOG_PATHS = {
    "server": Path("/kaggle/working/server.log"),
    "sd35": Path("/kaggle/working/sd35_worker.log"),
    "sam2_dino": Path("/kaggle/working/sam2_dino_worker.log"),
    "trellis": Path("/kaggle/working/trellis_worker.log"),
}
PROCESS_NAMES = {
    "server": "server_proc",
    "sd35": "sd35_worker_proc",
    "sam2_dino": "sam2_dino_worker_proc",
    "trellis": "trellis_worker_proc",
}
HEALTH_URLS = {
    "server": "http://127.0.0.1:8000/api/health",
    "sd35": "http://127.0.0.1:8001/health",
    "trellis": "http://127.0.0.1:8002/health",
    "sam2_dino": "http://127.0.0.1:8003/health",
}

def tail_log(path, line_count=60):
    if not path.is_file():
        return f"Chua co {path.name}"
    text = path.read_text(encoding="utf-8", errors="replace")
    return "\n".join(text.splitlines()[-line_count:]) or "(log rong)"

def latest_log_line(path):
    if not path.is_file():
        return "chua co log"
    lines = [line.strip() for line in path.read_text(encoding="utf-8", errors="replace").splitlines() if line.strip()]
    return (lines[-1] if lines else "log rong")[-240:]

def health_summary(name, url):
    try:
        data = requests.get(url, timeout=1.5).json()
        if data.get("ready"):
            return f"{name}=READY"
        error = str(data.get("error") or "loading").splitlines()[0]
        return f"{name}=WAIT ({error[-90:]})"
    except Exception as exc:
        return f"{name}=OFFLINE ({type(exc).__name__})"

def show_backend_logs(line_count=60, clear=False):
    if clear:
        clear_output(wait=True)
    print("SpatialFlow backend monitor", datetime.now().strftime("%H:%M:%S"))
    print("")

    gpu = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.used,memory.free,memory.total,utilization.gpu",
            "--format=csv,noheader",
        ],
        capture_output=True, text=True, check=False,
    )
    print("GPU:", gpu.stdout.strip() or gpu.stderr.strip() or "khong doc duoc")
    print("Workers:", " | ".join(health_summary(name, url) for name, url in HEALTH_URLS.items()))

    states = []
    for name, variable in PROCESS_NAMES.items():
        process = globals().get(variable)
        if process is None:
            state = "not-started"
        elif process.poll() is None:
            state = f"running pid={process.pid}"
        else:
            state = f"stopped exit={process.returncode}"
        states.append(f"{name}: {state}")
    print("Processes:", " | ".join(states))
    print("Latest activity:")
    for log_name, log_path in LOG_PATHS.items():
        print(f"  {log_name}: {latest_log_line(log_path)}")

    for log_name, log_path in LOG_PATHS.items():
        print(f"\n===== {log_name} :: {log_path} =====")
        print(tail_log(log_path, line_count))

def follow_backend_logs(interval=2, line_count=60):
    try:
        while True:
            show_backend_logs(line_count=line_count, clear=True)
            print(f"\nTu dong lam moi moi {interval}s - dung cell de thoat.")
            time.sleep(interval)
    except KeyboardInterrupt:
        show_backend_logs(line_count=line_count, clear=True)
        print("\nDa dung theo doi log.")

# Tu dong theo doi realtime sau khi backend da khoi dong.
FOLLOW_LIVE = True
if FOLLOW_LIVE:
    follow_backend_logs(interval=2, line_count=60)
else:
    show_backend_logs(line_count=80)
    print("\nDat FOLLOW_LIVE = True va chay lai cell de xem log realtime.")

# Khi can dung toan bo backend:
# server_proc.terminate()
# sd35_worker_proc.terminate()
# sam2_dino_worker_proc.terminate()
# trellis_worker_proc.terminate()
# ngrok.kill()
